In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import sys
import argparse
from pathlib import Path
import pandas as pd
from goatools.obo_parser import GODag
from goatools.go_enrichment import GOEnrichmentStudy
from scipy.stats import hypergeom

# --- Helper function for directory selection ---
def select_directory(prompt_message, candidates, allow_multiple=False):
    if not candidates:
        print(f"Error: No valid directories found to choose from.", file=sys.stderr)
        return None
    
    print(prompt_message)
    for i, candidate in enumerate(candidates):
        print(f"  [{i + 1:2d}] {candidate.name}")
    
    try:
        choice_str = input(f"\nEnter number(s) (1-{len(candidates)}): ")
        if not choice_str.strip():
            sys.exit("No selection made. Exiting.")
        
        if allow_multiple:
            indices = {int(i.strip()) - 1 for i in choice_str.split(',')}
            return [candidates[i] for i in sorted(list(indices)) if 0 <= i < len(candidates)]
        else:
            choice = int(choice_str.strip())
            if 1 <= choice <= len(candidates):
                return candidates[choice - 1]
            else:
                raise ValueError("Choice out of range.")
    except (ValueError, EOFError, KeyboardInterrupt):
        sys.exit("\nInvalid selection or operation cancelled.")

# --- Core Enrichment Functions ---
def run_go_enrichment(gene_list, background_list, term2gene_df, go_dag):
    gene2go = term2gene_df.groupby('Gene')['GO_ID'].apply(set).to_dict()
    goeaobj = GOEnrichmentStudy(
        background_list, gene2go, go_dag,
        propagate_counts=True, alpha=0.05, methods=['fdr_bh']
    )
    goea_results = goeaobj.run_study(gene_list, p_fdr_bh=1.0)
    significant_results = [r for r in goea_results if r.p_fdr_bh < 0.05 and r.study_count > 0]
    if not significant_results: return pd.DataFrame()
    
    return pd.DataFrame([
        {
            "ID": r.GO, "Description": r.name, "GeneRatio": f"{r.study_count}/{r.study_n}",
            "BgRatio": f"{r.pop_count}/{r.pop_n}", "pvalue": r.p_uncorrected,
            "p.adjust": r.p_fdr_bh,
            "geneID": '/'.join(sorted(list(r.study_items))), "Count": r.study_count,
        } for r in significant_results
    ])

def filter_terminal_go(enrich_df, go_dag):
    if enrich_df.empty: return enrich_df
    go_ids = set(enrich_df['ID'])
    all_ancestors = {go_id: go_dag[go_id].get_all_parents() for go_id in go_ids if go_id in go_dag}
    redundant_terms = {go_id for go_id in go_ids for other_id in (go_ids - {go_id}) if other_id in all_ancestors and go_id in all_ancestors[other_id]}
    return enrich_df[enrich_df['ID'].isin(go_ids - redundant_terms)]

def run_ipr_enrichment(gene_list, background_list, term2gene_df):
    universe_genes, study_genes = set(background_list), set(gene_list)
    term2gene_df = term2gene_df[term2gene_df['Gene'].isin(universe_genes)]
    term_to_genes = term2gene_df.groupby('IPR_ID')['Gene'].apply(set)
    M, N = len(universe_genes), len(study_genes)
    results = []
    for term_id, term_genes in term_to_genes.items():
        k = len(study_genes.intersection(term_genes))
        if k == 0: continue
        n = len(term_genes)
        p_val = hypergeom.sf(k - 1, M, n, N)
        if p_val < 0.05:
            results.append({
                "ID": term_id, "Description": "Unknown", "GeneRatio": f"{k}/{N}", "BgRatio": f"{n}/{M}",
                "pvalue": p_val, "geneID": '/'.join(sorted(list(study_genes.intersection(term_genes)))), "Count": k
            })
    return pd.DataFrame(results)

# --- Main Execution Block ---
def main():
    parser = argparse.ArgumentParser(description="Run GO and InterPro enrichment analysis.")
    parser.add_argument('--base-dir', type=str, default='.', help="Project root directory.")
    
    is_jupyter = any(arg.startswith('-f') for arg in sys.argv)
    if is_jupyter:
        print("[INFO] Jupyter environment detected. Running in interactive mode.")
        args = parser.parse_args([])
    else:
        args = parser.parse_args()

    project_root = Path(args.base_dir).resolve()
    
    sample_candidates = [d for d in project_root.iterdir() if d.is_dir() and (d / 'commands.txt').is_file()]
    sample_dir = select_directory("\n--- Step 1: Select Sample Directory ---", sample_candidates)
    if not sample_dir: sys.exit()

    taxon_candidates = sorted([d for d in sample_dir.iterdir() if d.is_dir() and (d.name.startswith('Genus_') or d.name.startswith('Species_'))])
    taxon_dir = select_directory(f"\n--- Step 2: Select Taxon Directory in '{sample_dir.name}' ---", taxon_candidates)
    if not taxon_dir: sys.exit()
    
    print(f"\n[INFO] Starting analysis for: {taxon_dir.name}")

    mode = 'genus' if taxon_dir.name.startswith('Genus_') else 'species'
    paths = {
        "gene_list": taxon_dir / "BLAST_specify" / f"result_{mode}.txt",
        "background_list": sample_dir / "background_list.txt",
        "go_annot": sample_dir / "go_gene.tsv",
        "ipr_annot": sample_dir / "ipr_gene.tsv",
        "go_obo": project_root / "data" / "go.obo",
        "ipr_entry": project_root / "data" / "entry.list.txt"
    }
    for name, path in paths.items():
        if not path.is_file():
            sys.exit(f"Error: Required file not found: '{path}'")
        
    output_dir = taxon_dir / "Enrichment_Analysis"
    output_dir.mkdir(exist_ok=True)
    
    gene_list = pd.read_csv(paths["gene_list"], header=None)[0].tolist()
    background_list = pd.read_csv(paths["background_list"], header=None)[0].tolist()
    go_annot_df = pd.read_csv(paths["go_annot"], sep='\t', header=None, names=["GO_ID", "Gene"])
    ipr_annot_df = pd.read_csv(paths["ipr_annot"], sep='\t', header=None, names=["IPR_ID", "Gene"])
    
    go_dag = GODag(str(paths["go_obo"]))
    go_annot_df['Namespace'] = go_annot_df['GO_ID'].map(lambda go_id: go_dag[go_id].namespace if go_id in go_dag else None)
    go_annot_df.dropna(subset=['Namespace'], inplace=True)
    
    # --- MODIFIED SECTION: Calculate and store both raw and filtered results ---

    print("\n[INFO] Running GO BP enrichment...")
    go_bp = go_annot_df[go_annot_df['Namespace'] == 'biological_process']
    res_bp_raw = run_go_enrichment(gene_list, background_list, go_bp, go_dag)
    res_bp_filtered = filter_terminal_go(res_bp_raw.copy(), go_dag)
    res_bp_raw['Category'] = 'BP'
    res_bp_filtered['Category'] = 'BP'
    
    print("[INFO] Running GO MF enrichment...")
    go_mf = go_annot_df[go_annot_df['Namespace'] == 'molecular_function']
    res_mf_raw = run_go_enrichment(gene_list, background_list, go_mf, go_dag)
    res_mf_filtered = filter_terminal_go(res_mf_raw.copy(), go_dag)
    res_mf_raw['Category'] = 'MF'
    res_mf_filtered['Category'] = 'MF'
    
    print("[INFO] Running GO CC enrichment...")
    go_cc = go_annot_df[go_annot_df['Namespace'] == 'cellular_component']
    res_cc_raw = run_go_enrichment(gene_list, background_list, go_cc, go_dag)
    res_cc_filtered = filter_terminal_go(res_cc_raw.copy(), go_dag)
    res_cc_raw['Category'] = 'CC'
    res_cc_filtered['Category'] = 'CC'

    print("[INFO] Running InterPro enrichment...")
    res_ipr_df = run_ipr_enrichment(gene_list, background_list, ipr_annot_df)
    if not res_ipr_df.empty:
        ipr_entry_df = pd.read_csv(paths["ipr_entry"], sep='\t', header=0, names=["IPR_ID", "ENTRY_TYPE", "ENTRY_NAME"])
        ipr_id_to_name = ipr_entry_df.set_index('IPR_ID')['ENTRY_NAME']
        res_ipr_df['Description'] = res_ipr_df['ID'].map(ipr_id_to_name).fillna("Unknown")

    # Prepare a dictionary with all results for saving
    results_to_save = {
        "GO_BP_filtered": res_bp_filtered,
        "GO_MF_filtered": res_mf_filtered,
        "GO_CC_filtered": res_cc_filtered,
        "ALL_GO_filtered": pd.concat([res_bp_filtered, res_mf_filtered, res_cc_filtered], ignore_index=True),
        "IPR": res_ipr_df,
        "GO_BP_raw": res_bp_raw,
        "GO_MF_raw": res_mf_raw,
        "GO_CC_raw": res_cc_raw,
        "ALL_GO_raw": pd.concat([res_bp_raw, res_mf_raw, res_cc_raw], ignore_index=True)
    }
    # --- END OF MODIFIED SECTION ---
    
    print(f"\n[INFO] Writing results to {output_dir}...")
    excel_path = output_dir / f"{taxon_dir.name}_enrichment.xlsx"
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        # Write filtered results first, then raw results
        for sheet_name in sorted(results_to_save.keys()):
            df = results_to_save[sheet_name]
            if not df.empty:
                df.to_excel(writer, sheet_name=sheet_name, index=False)
    print(f"  - Saved Excel file: {excel_path.name}")
    
    for name, df in results_to_save.items():
        if not df.empty:
            tsv_path = output_dir / f"{name}.tsv"
            df.to_csv(tsv_path, sep='\t', index=False)
            print(f"  - Saved TSV file: {tsv_path.name}")

    print(f"\n[INFO] Process completed successfully.")

if __name__ == "__main__":
    main()

[INFO] Jupyter environment detected. Running in interactive mode.

--- Step 1: Select Sample Directory ---
  [ 1] microbiome
  [ 2] pain_sensitivity
  [ 3] silva_pain_sensitivity
  [ 4] depression
  [ 5] level-7

Enter number(s) (1-5): 4

--- Step 2: Select Taxon Directory in 'depression' ---
  [ 1] Genus_Adlercreutzia
  [ 2] Genus_Aerococcus
  [ 3] Genus_Allobaculum
  [ 4] Genus_Anaerofustis
  [ 5] Genus_Anaerostipes
  [ 6] Genus_Anaerotruncus
  [ 7] Genus_Arthrobacter
  [ 8] Genus_Bacteroides
  [ 9] Genus_Blautia
  [10] Genus_Brachybacterium
  [11] Genus_Brevibacterium
  [12] Genus_Candidatus_Neoarthromitus
  [13] Genus_Clostridium
  [14] Genus_Coprobacillus
  [15] Genus_Coprococcus
  [16] Genus_Corynebacterium
  [17] Genus_Dehalobacterium
  [18] Genus_Desulfovibrio
  [19] Genus_Dietzia
  [20] Genus_Dorea
  [21] Genus_Enterococcus
  [22] Genus_Facklamia
  [23] Genus_Holdemania
  [24] Genus_Lactobacillus
  [25] Genus_Odoribacter
  [26] Genus_Oscillospira
  [27] Genus_Parabacteroides
 